# Implementing `Orthogonal Nonnegative Matrix Tri-Factorizations for Clustering` From Scratch

## Section 1: Foundations of Nonnegative Matrix Factorization (NMF)

### 1.1 The Intuition: From Data to Parts-Based Representation

Matrix factorization techniques are fundamental tools in machine learning and data analysis which are designed to decompose a complex data matrix into a product of lower-rank matrices.<br>
Methods like Principal Component Analysis or as we call PCA find orthogonal directions of maximum variance in the data. However the resulting factors often contain mixed positive and negative values which can complicate their interpretation.<br>
**Nonnegative Matrix Factorization (NMF)** offers a distinct and powerful alternative from PCA. Its core principle is the decomposition of a nonnegative data matrix into the product of two other nonnegative matrices. This constraint of nonnegativity is not just a typical mathematical constraint but it fundamentally changes the nature of the decomposition. Instead of finding abstract variance-maximizing components NMF learns an additive, "parts-based" representation of the data.<br>

A classic illustration involves decomposing a matrix of face images, where NMF can learn to identify constituent parts like eyes, noses, and mouths. The original faces can then be reconstructed by additively combining these learned parts. This inherent interpretability has made NMF a valuable technique across diverse fields, including:
- Text mining (e.g., topic modeling)
- Bioinformatics (e.g., gene expression analysis)
- Pattern recognition

### 1.2 Mathematical Formulation of Standard 2-Factor NMF
Formally the standard 2-factor NMF problem seeks to approximate a given nonnegative data matrix $X \in {R}^{p \times n}$ by the product of two lower-rank nonnegative matrices:$F \in {R}^{p \times k}$ and $G \in \mathbb{R}^{n \times k}.$ <br>
Here:
- $p$ is the number of features (e.g., words in a vocabulary)
- $n$ is the number of samples (e.g., documents)
- $k$ is the number of latent components or topics which is typically $k \ll \min(p, n).$

The approximation is expressed as $X \approx F G^T.$


The goal is to find the factors $F$ and $G$ that minimize the reconstruction error between the original matrix $X$ and its approximation $FG^T.$ The most common objective function for this purpose is the squared Frobenius norm of the difference:  $\min_{F \geq 0, G \geq 0} \|X - F G^T\|_F^2.$<br>
The Frobenius norm $\|A\|_F$ is the square root of the sum of the squares of all elements of $A$: $\|A\|_F = \sqrt{ \sum_{i,j} A_{ij}^2 }$ <br>
Minimizing this objective function is equivalent to minimizing the sum of squared errors between each element of $X$ and its reconstructed counterpart in $FG^T.$ The matrices $F$ and $G$ are often referred to as the basis (or "features") matrix and the coefficients (or "encodings") matrix respectively.


#### The Challenge of Non-Uniqueness

A significant challenge with the standard NMF formulation is the non-uniqueness of its solution. For any given solution pair $(F,G)$ there exists a large set of alternative solutions that produce the exact same reconstruction error.<br>
Specifically for any invertible matrix $A$ such that both $F A$ and $(G(A^{-1})^T$ remains non-negative then the pair $F A, (G(A^{-1})^T$ is also a valid solution<br> $X \approx (FA)(G(A^{-1})^T)^T = FAGA^{-1} = F G^T $<br>

This non-uniqueness is not just a mathematical inconvenience but also it strikes at the core of NMF's practical value. The primary appeal of NMF lies in its promise of interpretable factors. For example, in text analysis a column in the factor matrix $F$ might be interpreted as a specific **topic** due to its high weights on related words. This interpretation is only meaningful however only if if the factors are stable and unique.<br>
If an equally valid solution exists where that **topic** is smeared across multiple columns or mixed with other topics the initial interpretation becomes arbitrary and unreliable. The ambiguity of the solution undermines the very interpretability that motivates the use of NMF in the first place. This fundamental problem makes necessary the introduction of additional constraints to guide the factorization towards a more meaningful and unique solution.

### Implementation:

#### 1. Setting Up Python Environment

In [1]:
import numpy as np
import re
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer